In [0]:
from Notifications import configs,receiver_emails,cc_emails,webhooks

In [0]:
from app import convert_df_to_html,send_dataframe_in_body,send_chat_message,create_tabular_card

In [0]:
df = spark.sql("select * from dev.bronze.employee")
html = convert_df_to_html(df,"Employee Report")
if configs['email_notification']:    
    try:
        send_dataframe_in_body(
            html,
            sender_email="mxnagori@gmail.com",
            receiver_email=",".join(receiver_emails),
            cc_emails=cc_emails,
            subject="Spark Report Generate"
        )
    except Exception as e:
        send_dataframe_in_body(html_content=f"<h3>EMAIL SENDING CODE FAILED <p>{str(e)}</p></h3>",
                               sender_email="mxnagori@gmail.com",
                               receiver_email=",".join(receiver_emails),
                               cc_emails=cc_emails,
                               subject="Failure email sending code")

In [0]:
import pandas
df = spark.sql("select * from dev.bronze.employee")
files = df.toPandas().to_csv(index=True)
csv_file = files
# send_chat_message(csv_file)
print(csv_file)

In [0]:
import requests
import os
WEBHOOK_URL = ""
def send_file_to_chat(file_name):
  try:
    with open(file_name,'rb') as f:
      print(file_name)
      files = {'file':(os.path.basename(file_name),f,'text/csv')}
      print(files)
      payload = {'message' : 'Here is the CSV report.'}
      response = requests.post(WEBHOOK_URL,headers={'Content-Type':'application/json'},data=json.dumps(payload))
      print(response.text)
      return response.status_code
  except Exception as e:
    return requests.post(str(e))

In [0]:
df = spark.sql("select * from dev.bronze.employee")
output_path = "/Volumes/dev/bronze/external_vol/temp/"
csv_file  = df.coalesce(1).write.mode("overwrite").option("header", True).csv(output_path)
csv_file=None
for file in os.listdir(output_path):
    if file.endswith(".csv"):
        csv_file = f"{output_path}/{file}"
        break
print(csv_file)
# send_file_to_chat(csv_file)

In [0]:
df = spark.sql("""select * from dev.bronze.employee""")
temp = "/Volumes/file_upload/files/external_volume/temp/output/"
save_df = df.write.mode("overwrite").option("header",True).csv(temp)
csv_file=None
for file in os.listdir(temp):
    if file.endswith(".csv"):
        csv_file = f"{temp}/{file}"
        break

file_upload_to_google_chat('/XXXXX',csv_file,"Emloyee csv file attachment.")

In [0]:
import os
df = spark.sql("""select * from dev.bronze.order_raw""")
temp = "/Volumes/file_upload/files/external_volume/temp/output/"
save_df = df.coalesce(1).write.mode("overwrite").option("header",True).csv(temp)
csv_file=None
for file in os.listdir(temp):
    if file.endswith(".csv"):
        csv_file = f"{temp}/{file}"
        break
create_tabular_card(dbutils.secrets.get(scope="google_chat",key="dl_notification"),csv_file,"Order_Details")

In [0]:
df = spark.sql("""select * from dev.bronze.order_raw""")
create_tabular_card(csv_file,"Order_Details")

In [0]:
call_fun("dl_notification")

In [0]:
import os

def running_webhook(webhook_type):
    url = webhooks.get(webhook_type)
    if not url:
        print("No webhooks url found")
    if webhooks.get("group_1") == url:
        df = spark.sql("""select * from dev.bronze.order_raw""")
        temp = "/Volumes/file_upload/files/external_volume/temp/output/"
        save_df = df.coalesce(1).write.mode("overwrite").option("header",True).csv(temp)
        csv_file=None
        for file in os.listdir(temp):
            if file.endswith(".csv"):
                csv_file = f"{temp}/{file}"
                break
        create_tabular_card(webhooks.get("group_1"),csv_file,"Order_Details")
        print(f"Message have send group_1 chats {webhooks.get("group_1")}")
    elif webhooks.get("group_2") == url:
        df = spark.sql("""select * from dev.bronze.employee""")
        temp = "/Volumes/file_upload/files/external_volume/temp/output/"
        save_df = df.coalesce(1).write.mode("overwrite").option("header",True).csv(temp)
        csv_file=None
        for file in os.listdir(temp):
            if file.endswith(".csv"):
                csv_file = f"{temp}/{file}"
                break
        create_tabular_card(webhooks.get("group_2"),csv_file,"Employee_Information")
        print(f"Message have send group_2 chats")
    else:
        print("No webhooks url found")

In [0]:
running_webhook("group_2")

In [0]:
username = dbutils.secrets.get(scope="google_chat",key="dl_notification")
print(username)

In [0]:
from IPython.display import HTML, display

def create_tabular_card_readable(df, title, limit_rows=20):
    pdf = df.limit(limit_rows).toPandas()

    headers = pdf.columns.tolist()
    rows = pdf.astype(str).values.tolist()

    # Calculate column widths
    col_widths = [max(len(h), max(len(str(row[i])) for row in rows)) for i, h in enumerate(headers)]

    def format_row(row):
        return " | ".join(str(val).ljust(col_widths[i]) for i, val in enumerate(row))

    header_line = format_row(headers)
    separator = "-+-".join("-" * w for w in col_widths)
    data_lines = [format_row(row) for row in rows]

    table_text = (
        f"<b>{title}</b><br><br>"
        f"<pre>{header_line}<br>{separator}<br>"
        + "<br>".join(data_lines)
        + "</pre>"
    )

    card_payload = {
        "cardsV2": [
            {
                "cardId": "table-card",
                "card": {
                    "sections": [
                        {
                            "widgets": [
                                {"textParagraph": {"text": table_text}}
                            ]
                        }
                    ]
                }
            }
        ]
    }

    #display(HTML(f"<pre>{header_line}\n{separator}\n" + "\n".join(data_lines) + "</pre>"))
    return card_payload

In [0]:
from app import create_tabular_card_readable

In [0]:
from IPython.display import HTML, display

def create_tabular_card_readable(df, title, limit_rows=50):
    pdf = df.limit(limit_rows).toPandas()   # keep preview small

    headers = list(pdf.columns)
    data_rows = pdf.astype(str).values.tolist()

    table_html = f"""
    <div style="max-height:300px; overflow:auto; border:1px solid #ddd;">
      <table border="1" cellpadding="6" cellspacing="0" style="border-collapse: collapse; font-size:12px;">
        <tr>
          {''.join([f'<th style="background:#f5f5f5; position:sticky; top:0;">{h}</th>' for h in headers])}
        </tr>
        {''.join([
            '<tr>' + ''.join([f'<td>{cell}</td>' for cell in row]) + '</tr>'
            for row in data_rows
        ])}
      </table>
    </div>
    """

    card_payload = {
        "cardsV2": [
            {
                "cardId": "table-card",
                "card": {
                    "sections": [
                        {
                            "widgets": [
                                {
                                    "textParagraph": {
                                        "text": f"<b>{title}</b><br><br>{table_html}"
                                    }
                                }
                            ]
                        }
                    ]
                }
            }
        ]
    }

    display(HTML(table_html))

In [0]:
df = spark.sql("""select * from dev.bronze.employee""")
create_tabular_card_readable(df, "Order Details Preview")

In [0]:
def format_table_aligned(df, limit_rows=50):
    pdf = df.limit(limit_rows).toPandas().astype(str)

    headers = pdf.columns.tolist()
    rows = pdf.values.tolist()

    # calculate max width for each column (header vs data)
    col_widths = [
        max(len(headers[i]), max(len(row[i]) for row in rows))
        for i in range(len(headers))
    ]

    def fmt_row(row):
        return " | ".join(row[i].ljust(col_widths[i]) for i in range(len(row)))

    header_line = fmt_row(headers)
    separator = " | ".join("-" * col_widths[i] for i in range(len(col_widths)))
    data_lines = [fmt_row(row) for row in rows]

    table_text = "\n".join([header_line, separator] + data_lines)
    return table_text

In [0]:
df = spark.sql("""select * from dev.bronze.employee""")
format_table_aligned(df, "Order Details Preview")